# Preparación del dataset para modelado binario FIRMS

Este notebook construye el dataset utilizado en `Modelos.ipynb`. El objetivo supervisado es estimar si una combinación departamento-semana presenta al menos una detección FIRMS.

Los perfiles A–D se forman únicamente con variables meteorológicas. `cantidad_detecciones` no participa en el escalado ni en K-Means; se utiliza después para construir y auditar el target binario.

## 1. Librerías y rutas

Se trabaja con la base semanal integrada existente. El resultado se guarda separado de la fuente original.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

RUTA_ENTRADA = Path("data/processed/firms_open_meteo_weekly/open_meteo_firms_weekly_target_available.parquet")
RUTA_SALIDA = Path("data/processed/modelado/dataset_modelado_presencia_firms.parquet")

CLAVES = ["departamento", "fecha_inicio_semana"]
VARIABLES_METEOROLOGICAS = [
    "temperature_2m_max_mean_weekly",
    "relative_humidity_2m_min_mean_weekly",
    "wind_speed_10m_max_mean_weekly",
    "precipitation_sum_weekly",
]
PERFILES = ["A", "B", "C", "D"]

## 2. Carga y controles iniciales

Se comprueba que estén las columnas necesarias, que cada departamento-semana sea único y que FIRMS no tenga valores faltantes o negativos.

In [ ]:
datos = pd.read_parquet(RUTA_ENTRADA)
datos["fecha_inicio_semana"] = pd.to_datetime(datos["fecha_inicio_semana"], errors="raise")

columnas_requeridas = set(CLAVES + VARIABLES_METEOROLOGICAS + ["cantidad_detecciones", "anio"])
columnas_faltantes = sorted(columnas_requeridas.difference(datos.columns))

assert not columnas_faltantes, f"Faltan columnas requeridas: {columnas_faltantes}"
assert not datos.duplicated(CLAVES).any(), "Hay claves departamento-semana duplicadas"
assert not datos["cantidad_detecciones"].isna().any(), "Hay detecciones faltantes"
assert not datos["cantidad_detecciones"].lt(0).any(), "Hay detecciones negativas"

print("Dimensiones de la fuente:", datos.shape)
print("Período:", datos["fecha_inicio_semana"].min().date(), "a", datos["fecha_inicio_semana"].max().date())
print("Claves duplicadas:", int(datos.duplicated(CLAVES).sum()))

## 3. Split temporal

Se conserva la partición acordada: train 2018–2023, validation 2024 y test 2025. El test queda identificado, pero no se utiliza para evaluar modelos en este notebook.

In [ ]:
datos["split"] = np.select(
    [
        datos["anio"].between(2018, 2023),
        datos["anio"].eq(2024),
        datos["anio"].eq(2025),
    ],
    ["train", "validation", "test"],
    default="fuera_alcance",
)

assert not datos["split"].eq("fuera_alcance").any()
datos["split"].value_counts().reindex(["train", "validation", "test"])

## 4. Matriz meteorológica

K-Means utiliza solamente cuatro variables meteorológicas. Se aplica `log1p` a la precipitación para reducir su asimetría. El `StandardScaler` se ajusta exclusivamente con train para evitar fuga de información temporal.

In [ ]:
def preparar_matriz_meteorologica(tabla):
    matriz = tabla[VARIABLES_METEOROLOGICAS].astype(float).copy()
    assert not matriz.isna().any().any(), "La matriz meteorológica contiene nulos"
    assert not matriz["precipitation_sum_weekly"].lt(0).any(), "La precipitación contiene valores negativos"
    matriz["precipitation_sum_weekly"] = np.log1p(matriz["precipitation_sum_weekly"])
    return matriz

es_train = datos["split"].eq("train")
matriz_train = preparar_matriz_meteorologica(datos.loc[es_train])
matriz_completa = preparar_matriz_meteorologica(datos)

scaler = StandardScaler()
matriz_train_escalada = scaler.fit_transform(matriz_train)
matriz_completa_escalada = scaler.transform(matriz_completa)

print("Variables utilizadas por K-Means:")
print(VARIABLES_METEOROLOGICAS)

## 5. Ajuste de K-Means y perfiles A–D

K-Means se ajusta únicamente con 2018–2023. Luego el modelo congelado asigna perfiles a todas las observaciones mediante `transform` y `predict`.

Las letras no son niveles de riesgo. Para evitar depender del número arbitrario del cluster, los dos centroides más fríos se denominan A (más lluvioso) y B (más seco); los dos más cálidos se denominan C (más lluvioso) y D (más seco).

In [ ]:
kmeans = KMeans(n_clusters=4, random_state=42, n_init=20)
kmeans.fit(matriz_train_escalada)

centroides = pd.DataFrame(
    scaler.inverse_transform(kmeans.cluster_centers_),
    columns=VARIABLES_METEOROLOGICAS,
)
centroides["precipitation_sum_weekly"] = np.expm1(centroides["precipitation_sum_weekly"])

clusters_frios = centroides.nsmallest(2, "temperature_2m_max_mean_weekly").index.tolist()
clusters_calidos = centroides.nlargest(2, "temperature_2m_max_mean_weekly").index.tolist()
frio_humedo = centroides.loc[clusters_frios, "precipitation_sum_weekly"].idxmax()
frio_seco = next(cluster for cluster in clusters_frios if cluster != frio_humedo)
calido_humedo = centroides.loc[clusters_calidos, "precipitation_sum_weekly"].idxmax()
calido_seco = next(cluster for cluster in clusters_calidos if cluster != calido_humedo)

mapeo_perfiles = {
    int(frio_humedo): "A",
    int(frio_seco): "B",
    int(calido_humedo): "C",
    int(calido_seco): "D",
}

centroides["perfil_meteorologico"] = centroides.index.map(mapeo_perfiles)
centroides.sort_values("perfil_meteorologico")

In [ ]:
clusters_asignados = kmeans.predict(matriz_completa_escalada)
datos["perfil_meteorologico"] = pd.Categorical(
    pd.Series(clusters_asignados, index=datos.index).map(mapeo_perfiles),
    categories=PERFILES,
)

pd.crosstab(datos["split"], datos["perfil_meteorologico"], dropna=False)

## 6. Target binario y variables descriptivas

`presencia_firms` vale 1 si existe al menos una detección FIRMS y 0 en caso contrario. Se crean además mes y estación como variables de contexto. La cantidad original se conserva para trazabilidad, pero no se usa para formar los perfiles.

In [ ]:
datos["presencia_firms"] = datos["cantidad_detecciones"].gt(0).astype("int8")
datos["mes"] = datos["fecha_inicio_semana"].dt.month.astype("int8")

estacion_por_mes = {
    12: "Verano", 1: "Verano", 2: "Verano",
    3: "Otoño", 4: "Otoño", 5: "Otoño",
    6: "Invierno", 7: "Invierno", 8: "Invierno",
    9: "Primavera", 10: "Primavera", 11: "Primavera",
}
datos["estacion"] = pd.Categorical(
    datos["mes"].map(estacion_por_mes),
    categories=["Verano", "Otoño", "Invierno", "Primavera"],
)

pd.crosstab(datos["split"], datos["presencia_firms"])

## 7. Dataset final

Se seleccionan explícitamente las columnas necesarias para el modelado y se ordenan las observaciones por departamento y semana. No se modifica el archivo integrado de origen.

In [ ]:
columnas_salida = [
    "departamento", "fecha_inicio_semana", "anio", "mes", "estacion",
    *VARIABLES_METEOROLOGICAS,
    "perfil_meteorologico", "cantidad_detecciones", "presencia_firms", "split",
]

dataset_modelado = (
    datos[columnas_salida]
    .sort_values(CLAVES)
    .reset_index(drop=True)
)

dataset_modelado.head()

## 8. Validaciones finales

Estos controles reemplazan al test separado: verifican dimensiones, claves, nulos, definición del target, perfiles y tamaños de los splits antes de guardar el archivo. Si algo cambia de forma inesperada, la ejecución se detiene con un mensaje claro.

In [ ]:
assert dataset_modelado.shape == (7923, 13), f"Dimensión inesperada: {dataset_modelado.shape}"
assert not dataset_modelado.isna().any().any(), "El dataset final contiene valores faltantes"
assert not dataset_modelado.duplicated(CLAVES).any(), "El dataset final contiene claves duplicadas"
assert set(dataset_modelado["presencia_firms"].unique()) == {0, 1}
assert dataset_modelado["presencia_firms"].eq(
    dataset_modelado["cantidad_detecciones"].gt(0).astype("int8")
).all(), "El target binario no coincide con FIRMS"
assert set(dataset_modelado["perfil_meteorologico"].astype(str).unique()) == set(PERFILES)

splits_esperados = {"train": 5947, "validation": 1007, "test": 969}
assert dataset_modelado["split"].value_counts().to_dict() == splits_esperados

resumen_validacion = pd.DataFrame({
    "control": ["filas", "columnas", "duplicados", "nulos", "departamentos"],
    "resultado": [
        len(dataset_modelado),
        dataset_modelado.shape[1],
        int(dataset_modelado.duplicated(CLAVES).sum()),
        int(dataset_modelado.isna().sum().sum()),
        dataset_modelado["departamento"].nunique(),
    ],
})
resumen_validacion

## 9. Exportación

La exportación se realiza únicamente después de superar todos los controles anteriores.

In [ ]:
RUTA_SALIDA.parent.mkdir(parents=True, exist_ok=True)
dataset_modelado.to_parquet(RUTA_SALIDA, index=False)

print("Dataset guardado en:", RUTA_SALIDA)
print("Dimensiones:", dataset_modelado.shape)
print("Mapeo cluster a perfil:", mapeo_perfiles)